# Análise Numérica de Sistemas Lineares com a Matriz de Hilbert: Eliminação Gaussiana e Cálculo do Determinante

**Disciplina**: Cálculo Numérico  
**Professor**: Marcos Maia  
**Autor**: Yann Keven Jordão Leão (Engenharia da Computação - UFRPE)  

## Introdução

Nesta atividade, aplicaremos a interpolação polinomial de Newton para estimar o **calor específico da água a 37,5 °C**, com base nos dados da tabela a seguir:

| Temperatura (°C) | Calor específico |
|:----------------:|:----------------:|
| 30               | 0.99826          |
| 35               | 0.99818          |
| 40               | 0.99828          |
| 45               | 0.99849          |

Com esses pontos, construiremos um **polinômio interpolador de grau 3**. Em seguida, utilizaremos esse polinômio para resolver numericamente a equação $p(x) = 0{,}99837$, empregando o **método da secante** com erro absoluto inferior a $10^{-6}$, a fim de determinar a temperatura correspondente a esse valor de calor específico.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Valores da tabela
temperaturas = np.array([30, 35, 40, 45], dtype=float)
calores = np.array([0.99826, 0.99818, 0.99828, 0.99849], dtype=float)

# Gráfico de pontos
plt.figure(figsize=(5, 3)) 
plt.scatter(temperaturas, calores, color='red')
plt.xlabel('Temperatura (°C)')
plt.ylabel('Calor específico')
plt.title('Calor específico x Temperatura')
plt.grid(True)
plt.show()

## Interpolação Polinomial de Newton (Diferenças divididas)

Para esta atividade, optei pela **interpolação de Newton**, que utiliza as chamadas diferenças divididas para construir um polinômio interpolador. Essa abordagem tem vantagens computacionais em relação à fórmula de Lagrange, especialmente quando novos pontos precisam ser adicionados.

O polinômio interpolador é construído na forma:

$$
p(x) = f[x_0] + f[x_0, x_1](x - x_0) + f[x_0, x_1, x_2](x - x_0)(x - x_1) + \dots + f[x_0, x_1, \dots, x_n](x - x_0)\dots(x - x_{n-1})
$$

Onde cada termo $f[x_i, x_{i+1}, \dots]$ representa uma **diferença dividida**, calculada recursivamente com base nos valores da tabela. Definida da seguinte forma:

In [ ]:
# Função para a interpolação
def interpolacao_polinomial(x, y):
    n = len(x)
    tabela = np.zeros((n, n))
    tabela[:,0] = y  # primeira coluna é y

    # Calcula as diferenças divididas
    for coluna in range(1, n):
        for linha in range(n - coluna):
            tabela[linha][coluna] = (tabela[linha + 1][coluna - 1] - tabela[linha][coluna - 1]) / (x[linha + coluna] - x[linha])
    
    return tabela[0]  # retorna apenas a primeira linha (os coeficientes do polinômio)        

### Polinômio interpolador obtido

Os coeficientes das diferenças divididas, correspondentes ao polinômio de Newton, foram:



In [ ]:
# Coeficientes do polinômio de Newton
coef = interpolacao_polinomial(temperaturas, calores)
print("Coeficientes do polinômio de Newton:", coef)

O polinômio interpolador $p(x)$, de grau 3, fica então:

$$
\begin{aligned}
p(x) =\ & 9.9826 \times 10^{-1} \\
& - 1.6 \times 10^{-5}(x - 30) \\
& + 3.6 \times 10^{-6}(x - 30)(x - 35) \\
& - 9.333 \times 10^{-8}(x - 30)(x - 35)(x - 40)
\end{aligned}
$$

In [ ]:
# Pontos para plotar o polinômio suavemente
x_vals = np.linspace(30, 45, 200)
y_vals = [avalia_polinomio_newton(coef, temperaturas, x) for x in x_vals]

# Plot
plt.figure(figsize=(6, 4))
plt.scatter(temperaturas, calores, color='red', label='Pontos da tabela')
plt.plot(x_vals, y_vals, '--', color='red', label='Polinômio interpolador')
plt.xlabel('Temperatura (°C)')
plt.ylabel('Calor específico')
plt.title('Interpolação polinomial de Newton')
plt.grid(True)
plt.legend()
plt.show()

### Estimativa do calor específico da água a 37,5 °C

Substituindo $x = 37{,}5$ no polinômio interpolador, obtemos:

In [ ]:
# Calcular o x na interpolação
def avalia_polinomio_newton(coef, x_dados, x):
    n = len(coef)
    p = coef[0]
    for i in range(1, n):
        termo = coef[i]
        for j in range(i):
            termo *= (x - x_dados[j])
        p += termo
    return p

In [ ]:
# Calor específico a 37.5 °C
x_interp = 37.5
calor_interp = avalia_polinomio_newton(coef, temperaturas, x_interp)
print(f"Calor específico aproximado em {x_interp} °C: {round(calor_interp, 5)}")

Portanto, a estimativa para o calor específico da água a 37,5 °C é:

$$
p(37{,}5) \approx 0{,}99821
$$

Esse valor faz sentido dentro do comportamento dos dados originais, e reforça a adequação da interpolação polinomial de grau 3 ao problema.

## Determinação da temperatura correspondente a um calor específico conhecido

Com o polinômio interpolador $p(x)$ já construído, agora buscamos resolver a equação:

$$
p(x) = 0{,}99837
$$

Isso equivale a encontrar o valor de temperatura $x$ tal que o calor específico da água seja **0.99837**. Para isso, reescrevemos o problema como:

$$
f(x) = p(x) - 0{,}99837 = 0
$$

Como o polinômio $p(x)$ é conhecido, podemos resolver essa equação numericamente utilizando um método iterativo.

In [ ]:
# Valor alvo de calor específico
valor_alvo = 0.99837

# Função f(x) = p(x) - valor_alvo
def f(x):
    return avalia_polinomio_newton(coef, temperaturas, x) - valor_alvo

# Tolerância
tolerance = 1e-6

### Método de resolução

Optei pelo **método da secante**, pois é rápido para convergir e é uma alternativa ao método de Newton-Raphson, já que não requer o cálculo da derivada da função.
O processo iterativo será repetido até que o erro absoluto entre iterações consecutivas ou o valor da função seja inferior a uma tolerância de $10^{-6}$. Escolhi os limites das temperaturas, logo: $x_0 = 30$ e $x_1 = 45$

O resultado nos fornece a **temperatura estimada** para o calor específico desejado.

In [ ]:
# Método da secante
def secante(x0, x1, tolerance=1e-6, max_iterations=100):
    if abs(f(x0)) < tolerance:
        return {"result": float(x0), "iterations": 0}
    
    if abs(f(x1)) < tolerance or abs(x1 - x0) < tolerance:
        return {"result": float(x1), "iterations": 0}

    for iteration in range(1, max_iterations + 1):
        denominator = f(x1) - f(x0)
        if denominator == 0:
            print("Divisão por zero detectada na iteração", iteration)
            return {"result": float(x1), "iterations": iteration}

        x2 = x1 - f(x1) * (x1 - x0) / denominator

        if abs(f(x2)) < tolerance or abs(x2 - x1) < tolerance:
            return {"result": float(x2), "iterations": iteration}

        x0, x1 = x1, x2

    print("Número máximo de iterações atingido")
    return {"result": float(x2), "iterations": max_iterations}

In [ ]:
temperatura_estimada = secante(30, 45)
print(f"Temperatura aproximada: {temperatura_estimada['result']:.6f} °C (em {temperatura_estimada['iterations']} iterações)")

## Conclusão

Através da interpolação polinomial de Newton, obtivemos um polinômio de grau 3 que aproxima o comportamento do calor específico da água em função da temperatura.

- Com esse polinômio, estimamos o **calor específico da água a 37,5 °C** como aproximadamente **0,99821**, valor coerente com a tendência observada nos dados fornecidos.
- Em seguida, resolvemos numericamente a equação $p(x) = 0{,}99837$, utilizando o **método da secante**, que dispensou o cálculo da derivada do polinômio.

O resultado do método indicou que a **temperatura correspondente a um calor específico de 0,99837 é aproximadamente 42,361 °C**, respeitando o critério de tolerância de $10^{-6}$.

Esses procedimentos demonstram a eficácia da interpolação polinomial e dos métodos numéricos na obtenção de estimativas precisas a partir de um conjunto discreto de dados experimentais.
